In [0]:

from pyspark.sql import functions as F

# Carregar perfis traduzidos
perfis = spark.table(
    "workspace.silver.perfis_traduzidos"
)

# Função para contar respostas, incluindo arrays nulos
def qtd_respostas(campo):
    return F.coalesce(
        F.size(F.col(campo + "_codigos")),
        F.lit(0)
    )

# Adicionar indicadores de qualidade
perfis_auditados = (
    perfis
    .withColumn(
        "qtd_respostas_orcamento",
        qtd_respostas("form_b")
    )
    .withColumn(
        "orcamento_todas_faixas",
        qtd_respostas("form_b") == 4
    )
    .withColumn(
        "qtd_respostas_popularidade",
        qtd_respostas("form_j")
    )
    .withColumn(
        "popularidade_todas_opcoes",
        qtd_respostas("form_j") == 3
    )
    .withColumn(
        "regiao_sem_resposta",
        qtd_respostas("form_r") == 0
    )
    .withColumn(
        "regiao_inconsistente",
        (
            F.array_contains(
                F.col("form_r_codigos"), "0"
            ) &
            (qtd_respostas("form_rr") > 0)
        ) |
        (
            F.array_contains(
                F.col("form_r_codigos"), "1"
            ) &
            (qtd_respostas("form_rr") == 0)
        )
    )
    .withColumn(
        "regiao_inconsistente",
        F.coalesce(
            F.col("regiao_inconsistente"),
            F.lit(False)
        )
    )
    .withColumn(
        "status_regiao",
        F.when(
            F.col("regiao_sem_resposta"),
            "Não informado"
        )
        .when(
            F.col("regiao_inconsistente"),
            "Inconsistente"
        )
        .otherwise("Consistente")
    )
)

# Salvar nova tabela Silver
(
    perfis_auditados.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.perfis_auditados")
)

print("Tabela Silver auditada criada!")
print("Viajantes:", perfis_auditados.count())
print("Colunas:", len(perfis_auditados.columns))


In [0]:

from pyspark.sql import functions as F

auditados = spark.table(
    "workspace.silver.perfis_auditados"
)

print("STATUS DAS PREFERÊNCIAS GEOGRÁFICAS")

display(
    auditados
    .groupBy("status_regiao")
    .count()
    .orderBy("status_regiao")
)

print("INDICADORES DE QUALIDADE")

display(
    auditados.agg(
        F.sum(
            F.col("orcamento_todas_faixas").cast("int")
        ).alias("orcamento_todas_faixas"),
        F.sum(
            F.col("popularidade_todas_opcoes").cast("int")
        ).alias("popularidade_todas_opcoes"),
        F.sum(
            F.col("regiao_inconsistente").cast("int")
        ).alias("regioes_inconsistentes")
    )
)


In [0]:

from pyspark.sql import functions as F

perfis = spark.table("workspace.silver.perfis_traduzidos")

campos = [
    "form_a", "form_b", "form_c", "form_f",
    "form_g", "form_h", "form_i", "form_j",
    "form_r", "form_rr"
]

resultado = []

for campo in campos:
    dados = perfis.agg(
        F.count(F.when(
            F.col(campo).isNotNull() &
            F.col(campo + "_codigos").isNull(), 1
        )).alias("erros_leitura"),
        F.count(F.when(
            F.exists(
                F.col(campo + "_pt"),
                lambda x: x.startswith("Código desconhecido:")
            ), 1
        )).alias("codigos_desconhecidos")
    ).first()

    resultado.append((
        campo,
        dados["erros_leitura"],
        dados["codigos_desconhecidos"]
    ))

display(
    spark.createDataFrame(
        resultado,
        ["campo", "erros_leitura", "codigos_desconhecidos"]
    )
)


In [0]:

perfis = spark.table("workspace.silver.perfis_traduzidos")

print("Total de viajantes:", perfis.count())
print("Total de colunas:", len(perfis.columns))

display(
    perfis.select(
        "id",
        "form_a", "form_a_pt",
        "form_b", "form_b_pt",
        "form_c", "form_c_pt",
        "form_f", "form_f_pt",
        "form_g", "form_g_pt",
        "form_h", "form_h_pt",
        "form_i", "form_i_pt",
        "form_j", "form_j_pt",
        "form_r", "form_r_pt",
        "form_rr", "form_rr_pt"
    ).limit(20)
)


In [0]:

from pyspark.sql import functions as F

viajantes = spark.table("workspace.bronze.viajantes")

dicionarios = {
    "form_a": {
        "0": "0-19 anos", "1": "20-39 anos",
        "2": "40-59 anos", "3": "60+ anos"
    },
    "form_b": {
        "0": "0-49", "1": "50-99",
        "2": "100-249", "3": "300+"
    },
    "form_c": {
        "0": "Inverno", "1": "Primavera",
        "2": "Verão", "3": "Outono"
    },
    "form_f": {
        "0": "Praia", "1": "Aventura",
        "2": "Natureza", "3": "Cultura",
        "4": "Vida noturna", "5": "História",
        "6": "Compras", "7": "Gastronomia"
    },
    "form_g": {
        "0": "Urbana", "1": "Rural",
        "2": "Mar", "3": "Montanha",
        "4": "Lago", "5": "Deserto",
        "6": "Planícies", "7": "Selva"
    },
    "form_h": {
        "0": "Tranquilo e relaxante",
        "1": "Equilibrado", "2": "Ativo"
    },
    "form_i": {
        "0": "Muito preocupado com segurança",
        "1": "Equilibrado",
        "2": "Preparado para qualquer situação"
    },
    "form_j": {
        "0": "Fora dos roteiros tradicionais",
        "1": "Destino clássico",
        "2": "Popular e em alta"
    },
    "form_r": {
        "0": "Qualquer lugar",
        "1": "Regiões específicas"
    },
    "form_rr": {
        "e": "Europa", "n": "América do Norte",
        "c": "Caribe", "a": "Ásia",
        "s": "América do Sul",
        "m": "Oriente Médio", "f": "África",
        "o": "Oceania"
    }
}

perfis = viajantes

for campo, respostas in dicionarios.items():
    # As listas usam aspas simples no CSV
    # Normalizar para JSON válido
    texto = F.regexp_replace(
        F.col(campo).cast("string"), "'", '"'
    )

    codigos = F.from_json(texto, "array<string>")

    # Preservar os códigos como array
    perfis = perfis.withColumn(
        campo + "_codigos", codigos
    )

    # Criar mapa de tradução
    elementos = []
    for codigo, descricao in respostas.items():
        elementos.extend([F.lit(codigo), F.lit(descricao)])

    mapa = F.create_map(*elementos)

    # Traduzir todas as alternativas selecionadas
    perfis = perfis.withColumn(
        campo + "_pt",
        F.transform(
            codigos,
            lambda x: F.coalesce(
                F.element_at(mapa, x),
                F.concat(F.lit("Código desconhecido: "), x)
            )
        )
    )

# Salvar todos os perfis e campos originais
(
    perfis.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.perfis_traduzidos")
)

print("Perfis traduzidos:", perfis.count())
print("Total de colunas:", len(perfis.columns))

display(
    perfis.select(
        "id",
        "form_a_pt",
        "form_b_pt",
        "form_c_pt",
        "form_f_pt",
        "form_g_pt",
        "form_h_pt",
        "form_i_pt",
        "form_j_pt",
        "form_r_pt",
        "form_rr_pt"
    )
)


In [0]:

from pyspark.sql import functions as F

df = spark.table(
    "workspace.silver.avaliacoes_consolidadas"
)

print("Total consolidado:", df.count())

duplicados = (
    df.groupBy("usuario_id", "destino_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicados restantes:", duplicados)

display(
    df.groupBy("avaliacao")
    .count()
    .orderBy("avaliacao")
)


In [0]:

from pyspark.sql import functions as F

# Silver original, sem alterações
silver = spark.table(
    "workspace.silver.avaliacoes_com_localizacao"
)

# Identificar pares com respostas conflitantes
conflitos = (
    silver
    .groupBy("usuario_id", "destino_id")
    .agg(
        F.countDistinct("avaliacao").alias("tipos_avaliacao"),
        F.collect_set("avaliacao").alias("respostas")
    )
    .filter(F.col("tipos_avaliacao") > 1)
)

# Salvar conflitos para auditoria
(
    conflitos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.conflitos_avaliacoes")
)

# Remover pares conflitantes da base consolidada
silver_sem_conflitos = silver.join(
    conflitos.select("usuario_id", "destino_id"),
    on=["usuario_id", "destino_id"],
    how="left_anti"
)

# Manter uma ocorrência por usuário e destino
silver_consolidada = (
    silver_sem_conflitos
    .dropDuplicates(["usuario_id", "destino_id"])
)

# Salvar nova tabela Silver
(
    silver_consolidada.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.avaliacoes_consolidadas")
)

print("Silver consolidada criada!")
print("Pares conflitantes:", conflitos.count())
print("Avaliações consolidadas:", silver_consolidada.count())

display(
    silver_consolidada.select(
        "usuario_id",
        "destino_id",
        "cidade",
        "pais",
        "avaliacao",
        "form_a",
        "form_b",
        "form_c",
        "form_f"
    )
)


In [0]:

from pyspark.sql import functions as F

silver = spark.table(
    "workspace.silver.avaliacoes_com_localizacao"
)

# Identificar pares repetidos e suas avaliações
diagnostico = (
    silver
    .groupBy("usuario_id", "destino_id", "destino")
    .agg(
        F.count("*").alias("quantidade"),
        F.countDistinct("avaliacao").alias("tipos_avaliacao"),
        F.collect_set("avaliacao").alias("avaliacoes")
    )
    .filter(F.col("quantidade") > 1)
)

display(
    diagnostico.orderBy(
        F.desc("tipos_avaliacao"),
        F.desc("quantidade")
    )
)

# Resumo dos tipos de duplicidade
display(
    diagnostico.groupBy("tipos_avaliacao")
    .agg(
        F.count("*").alias("pares"),
        F.sum(F.col("quantidade") - 1).alias("linhas_excedentes")
    )
    .orderBy("tipos_avaliacao")
)


In [0]:

from pyspark.sql import functions as F

silver = spark.table(
    "workspace.silver.avaliacoes_com_localizacao"
)

# Verificar localizações ausentes
localizacoes_nulas = silver.filter(
    F.col("cidade").isNull() |
    F.col("pais").isNull() |
    (F.col("cidade") == "") |
    (F.col("pais") == "")
).count()

# Verificar avaliações duplicadas
duplicados = (
    silver
    .groupBy("usuario_id", "destino_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Localizações ausentes:", localizacoes_nulas)
print("Pares usuário-destino duplicados:", duplicados)


In [0]:

from pyspark.sql import functions as F

silver = spark.table(
    "workspace.silver.avaliacoes_com_localizacao"
)

# Validar integridade da Silver
resultado = silver.agg(
    F.count("*").alias("total_avaliacoes"),
    F.countDistinct("usuario_id").alias("total_viajantes"),
    F.countDistinct("destino_id").alias("destinos_avaliados"),
    F.sum(
        F.when(F.col("destino").isNull(), 1).otherwise(0)
    ).alias("destinos_sem_correspondencia"),
    F.sum(
        F.when(
            F.col("cidade").isNull() | F.col("pais").isNull(),
            1
        ).otherwise(0)
    ).alias("localizacoes_nulas")
)

display(resultado)


In [0]:

df = spark.table(
    "workspace.silver.avaliacoes_com_localizacao"
)

print("Total de avaliações:", df.count())

print(
    "Destinos sem nome:",
    df.filter(F.col("destino").isNull()).count()
)

print(
    "Cidades sem identificação:",
    df.filter(
        F.col("cidade").isNull() |
        (F.col("cidade") == "")
    ).count()
)

print(
    "Países sem identificação:",
    df.filter(
        F.col("pais").isNull() |
        (F.col("pais") == "")
    ).count()
)


In [0]:

from pyspark.sql import functions as F

df = spark.table(
    "workspace.silver.avaliacoes_com_localizacao"
)

# Resumo por país e cidade
resumo = (
    df.groupBy("pais", "cidade")
    .agg(
        F.count("*").alias("total_avaliacoes"),
        F.sum(
            F.when(F.col("avaliacao") == "Sim", 1).otherwise(0)
        ).alias("total_sim"),
        F.sum(
            F.when(F.col("avaliacao") == "Não", 1).otherwise(0)
        ).alias("total_nao"),
        F.sum(
            F.when(F.col("avaliacao") == "Talvez", 1).otherwise(0)
        ).alias("total_talvez")
    )
    .withColumn(
        "taxa_aceitacao_pct",
        F.round(
            F.col("total_sim") /
            F.when(
                F.col("total_sim") + F.col("total_nao") > 0,
                F.col("total_sim") + F.col("total_nao")
            ) * 100,
            2
        )
    )
    .orderBy(F.desc("total_sim"))
)

display(resumo)


In [0]:

from pyspark.sql import functions as F

# Ler a Silver já criada
df_silver = spark.table(
    "workspace.silver.avaliacoes_tratadas"
)

# Separar cidade e país a partir do nome do destino
# O país é o último elemento após a vírgula.
# A cidade é o primeiro elemento.
df_silver_completo = (
    df_silver
    .withColumn(
        "cidade",
        F.trim(F.split(F.col("destino"), ",").getItem(0))
    )
    .withColumn(
        "pais",
        F.trim(
            F.element_at(
                F.split(F.col("destino"), ","),
                -1
            )
        )
    )
)

# Salvar a Silver enriquecida
(
    df_silver_completo.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.silver.avaliacoes_com_localizacao"
    )
)

print("Silver com cidade e país criada com sucesso!")

display(
    df_silver_completo.select(
        "usuario_id",
        "destino_id",
        "destino",
        "cidade",
        "pais",
        "avaliacao",
        "form_a",
        "form_b",
        "form_c",
        "form_f"
    )
)


In [0]:

# Carregar a tabela Silver completa
df_silver = spark.table(
    "workspace.silver.avaliacoes_tratadas"
)

# Exibir a tabela no Databricks
display(df_silver)


In [0]:

from pyspark.sql import functions as F

silver = spark.table(
    "workspace.silver.avaliacoes_tratadas"
)

# Distribuição completa das avaliações
display(
    silver.groupBy("avaliacao")
    .agg(F.count("*").alias("quantidade"))
    .orderBy("avaliacao")
)

# Indicadores de qualidade
print("Total de registros:", silver.count())

print(
    "Usuários distintos:",
    silver.select("usuario_id").distinct().count()
)

print(
    "Destinos avaliados distintos:",
    silver.select("destino_id").distinct().count()
)

print(
    "Destinos sem correspondência:",
    silver.filter(F.col("destino").isNull()).count()
)

print(
    "Pares usuário-destino duplicados:",
    silver.groupBy("usuario_id", "destino_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)


In [0]:

from pyspark.sql import functions as F

# Ler as bases completas da Bronze
viajantes = spark.table("workspace.bronze.viajantes")
destinos = spark.table("workspace.bronze.destinos")

# Converter listas de texto em arrays
def converter_lista(nome_coluna):
    return F.from_json(
        F.col(nome_coluna),
        "array<int>"
    )

# Preparar todos os viajantes
viajantes_tratados = (
    viajantes
    .withColumn(
        "yes_array",
        converter_lista("yes_swipes")
    )
    .withColumn(
        "no_array",
        converter_lista("no_swipes")
    )
    .withColumn(
        "maybe_array",
        converter_lista("maybe_swipes")
    )
)

# Transformar todas as avaliações em linhas
sim = (
    viajantes_tratados
    .select(
        "*",
        F.explode("yes_array").alias("destino_id")
    )
    .withColumn("avaliacao", F.lit("Sim"))
)

nao = (
    viajantes_tratados
    .select(
        "*",
        F.explode("no_array").alias("destino_id")
    )
    .withColumn("avaliacao", F.lit("Não"))
)

talvez = (
    viajantes_tratados
    .select(
        "*",
        F.explode("maybe_array").alias("destino_id")
    )
    .withColumn("avaliacao", F.lit("Talvez"))
)

# Reunir TODAS as avaliações
avaliacoes = (
    sim
    .unionByName(nao)
    .unionByName(talvez)
)

# Relacionar com TODOS os destinos
silver = (
    avaliacoes
    .join(destinos, on="destino_id", how="left")
    .withColumnRenamed("id", "usuario_id")
    .drop("yes_array", "no_array", "maybe_array")
)

# Salvar a Silver completa
(
    silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.avaliacoes_tratadas")
)

print("SILVER COMPLETA CRIADA!")

print(
    "Total de avaliações:",
    spark.table(
        "workspace.silver.avaliacoes_tratadas"
    ).count()
)


In [0]:

from pyspark.sql import functions as F

# 1. Ler as tabelas Bronze
viajantes = spark.table("workspace.bronze.viajantes")
destinos = spark.table("workspace.bronze.destinos")

# 2. Converter as listas de texto em arrays de inteiros
def converter_lista(coluna):
    return F.from_json(
        F.col(coluna),
        "array<int>"
    )

# 3. Transformar cada lista em avaliações individuais
sim = (
    viajantes
    .select(
        F.col("id").alias("usuario_id"),
        F.explode(converter_lista("yes_swipes")).alias("destino_id")
    )
    .withColumn("avaliacao", F.lit("Sim"))
)

nao = (
    viajantes
    .select(
        F.col("id").alias("usuario_id"),
        F.explode(converter_lista("no_swipes")).alias("destino_id")
    )
    .withColumn("avaliacao", F.lit("Não"))
)

talvez = (
    viajantes
    .select(
        F.col("id").alias("usuario_id"),
        F.explode(converter_lista("maybe_swipes")).alias("destino_id")
    )
    .withColumn("avaliacao", F.lit("Talvez"))
)

# 4. Unir todas as avaliações
avaliacoes = sim.unionByName(nao).unionByName(talvez)

# 5. Relacionar os IDs com os nomes dos destinos
silver = (
    avaliacoes
    .join(destinos, on="destino_id", how="left")
)

# 6. Salvar como tabela Delta
(
    silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.avaliacoes_tratadas")
)

print("Tabela Silver criada com sucesso!")

# 7. Conferir os resultados
display(silver.limit(10))

display(
    silver.groupBy("avaliacao")
    .count()
    .orderBy("avaliacao")
)
